<a href="https://colab.research.google.com/github/wundertater/FerroCalc/blob/master/pinns_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q deepxde

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 9.0 MB/s eta 0:00:00


In [2]:
import os
os.environ["DDE_BACKEND"] = "pytorch"

import time
import random
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import deepxde as dde

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
dde.config.set_random_seed(SEED)

dde.config.set_default_float("float32")

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Set the default float type to float32


In [3]:
t = torch.linspace(0, 5, 500)

Аналитическое решение

In [4]:
def analytical_solution(t):
    """
    Exact solution of:
        x' + 2x + y = 0
        y' + x + 2y = 0

    x(0) = 1
    y(0) = 0
    """
    x = 0.5 * torch.exp(-t) + 0.5 * torch.exp(-3.0 * t)
    y = -0.5 * torch.exp(-t) + 0.5 * torch.exp(-3.0 * t)

    return torch.cat([x, y], dim=1)

In [5]:
#simple
class SimplePinn(nn.Module):
    def __init__(self):
        super().__init__()

        activation = nn.Tanh()

        self.first_layer = nn.Sequential(nn.Linear(1, 64),
                                         activation)
        self.hidden_layer1 = nn.Sequential(nn.Linear(64, 64),
                                         activation)
        self.hidden_layer2 = nn.Sequential(nn.Linear(64, 64),
                                         activation)
        self.last_layer = nn.Sequential(nn.Linear(64, 2))

    def forward(self, x):
        x = self.first_layer(x)
        x = self.hidden_layer1(x)
        x = self.hidden_layer2(x)
        x = self.last_layer(x)

        return x

In [6]:
class ResidualBlock(nn.Module):
    def __init__(self, width, activation):
        super().__init__()
        self.fc1 = nn.Linear(width, width)
        self.activation = activation

    def forward(self, x):
        return self.activation(x + self.activation(self.fc1(x)))


class PINN(nn.Module):
    def __init__(self, dim_hidden, num_layers=4):
        super().__init__()

        activation1 = nn.Tanh()

        self.hidden_layer = nn.Sequential(*[ResidualBlock(dim_hidden, activation1)
                                                for _ in range(num_layers)])

        self.first_layer = nn.Sequential(nn.Linear(2, dim_hidden),
                                         activation1)

        self.last_layer = nn.Sequential(nn.Linear(dim_hidden, 1))

    def forward(self, x):
        out = torch.cat([torch.sin(x), torch.cos(x)], dim=1)

        out = self.first_layer(out)
        out = self.hidden_layer(out)
        out = self.last_layer(out)

        return out

In [7]:
def loss_fn(model, t):
    output = model(t)
    x = output[:, 0:1]
    y = output[:, 1:2]

    dx_dt = torch.autograd.grad(
    outputs=x,
    inputs=t,
    grad_outputs=torch.ones_like(x),
    create_graph=True)[0]

    dy_dt = torch.autograd.grad(
    outputs=y,
    inputs=t,
    grad_outputs=torch.ones_like(y),
    create_graph=True)[0]

    res_x = dx_dt + 2*x + y
    res_y = dy_dt + x + 2*y

    init_loss_x = torch.square(x[0] - 1)
    init_loss_y = torch.square(y[0])

    loss = torch.mean(torch.square(res_x)) + torch.mean(torch.square(res_y)) + init_loss_x + init_loss_y
    return loss

In [8]:
def train(model, t, epochs, optimizer):
    for epoch in range(1, epochs+1):
        loss = loss_fn(model, t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print(f'Epoch: {epoch}, loss: {loss.item()}')

In [9]:
pinn = SimplePinn()
optimizer = torch.optim.Adam(pinn.parameters(), lr=0.001)

In [10]:
t_points = torch.tensor(t[:, None])
t_points.requires_grad_(True);

/tmp/ipykernel_494/3543813239.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_points = torch.tensor(t[:, None])


In [11]:
train(pinn, t_points, 5000, optimizer)

Epoch: 500, loss: 0.00010958099301205948
Epoch: 1000, loss: 2.720902193686925e-05
Epoch: 1500, loss: 1.5571085896226577e-05
Epoch: 2000, loss: 9.525138011667877e-06
Epoch: 2500, loss: 6.443673100875458e-06
Epoch: 3000, loss: 4.480597453948576e-06
Epoch: 3500, loss: 0.00016802118625491858
Epoch: 4000, loss: 4.1577335650799796e-05
Epoch: 4500, loss: 2.6522463940636953e-06
Epoch: 5000, loss: 2.1122039015608607e-06


In [12]:
def calculate_metrics(y_true, y_pred):
    error = y_pred - y_true

    # Абсолютная ошибка
    abs_error = torch.abs(error)

    # MAE
    mae = torch.mean(abs_error)

    # RMSE
    rmse = torch.sqrt(torch.mean(error ** 2))

    # Максимальная абсолютная ошибка
    max_error = torch.max(abs_error)

    # Relative L2 error
    relative_l2 = (
        torch.linalg.norm(error)
        / torch.linalg.norm(y_true)
    )

    # Relative L2 error отдельно для x и y
    relative_l2_x = (
        torch.linalg.norm(error[:, 0])
        / torch.linalg.norm(y_true[:, 0])
    )

    relative_l2_y = (
        torch.linalg.norm(error[:, 1])
        / torch.linalg.norm(y_true[:, 1])
    )

    # R²
    ss_res = torch.sum(error ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true, dim=0)) ** 2)

    r2 = 1.0 - ss_res / ss_tot

    # Ошибка в начальном условии
    initial_error = torch.abs(error[0])

    metrics = {
        "Relative L2": relative_l2.item(),
        "RMSE": rmse.item(),
        "MAE": mae.item(),
        "Max Absolute Error": max_error.item(),
        "Relative L2 x": relative_l2_x.item(),
        "Relative L2 y": relative_l2_y.item(),
        "R2": r2.item(),
        "Initial Error x": initial_error[0].item(),
        "Initial Error y": initial_error[1].item(),
    }

    return metrics

In [13]:
def physics_metrics(model, t):
    model.eval()

    t = t.clone().detach().requires_grad_(True)

    output = model(t)

    x = output[:, 0:1]
    y = output[:, 1:2]

    dx_dt = torch.autograd.grad(
        outputs=x,
        inputs=t,
        grad_outputs=torch.ones_like(x),
        create_graph=False
    )[0]

    dy_dt = torch.autograd.grad(
        outputs=y,
        inputs=t,
        grad_outputs=torch.ones_like(y),
        create_graph=False
    )[0]

    residual_x = dx_dt + 2.0 * x + y
    residual_y = dy_dt + x + 2.0 * y

    physics_mse_x = torch.mean(residual_x ** 2)
    physics_mse_y = torch.mean(residual_y ** 2)

    physics_mse = physics_mse_x + physics_mse_y

    physics_rmse = torch.sqrt(physics_mse)

    max_residual = torch.max(
        torch.cat([
            torch.abs(residual_x),
            torch.abs(residual_y)
        ])
    )

    return {
        "Physics MSE": physics_mse.item(),
        "Physics RMSE": physics_rmse.item(),
        "Physics MSE x": physics_mse_x.item(),
        "Physics MSE y": physics_mse_y.item(),
        "Max Physics Residual": max_residual.item(),
    }

In [14]:
def initial_condition_metrics(model):
    t0 = torch.tensor([[0.0]], dtype=torch.float32)

    model.eval()

    with torch.no_grad():
        prediction = model(t0)

    x_pred = prediction[0, 0].item()
    y_pred = prediction[0, 1].item()

    x_error = abs(x_pred - 1.0)
    y_error = abs(y_pred - 0.0)

    return {
        "x(0) prediction": x_pred,
        "y(0) prediction": y_pred,
        "x(0) error": x_error,
        "y(0) error": y_error,
        "IC MSE": (x_error**2 + y_error**2) / 2
    }

In [15]:
def evaluate_pinn(model, t_test):
    model.eval()

    with torch.no_grad():
        prediction = model(t_test)
        exact = analytical_solution(t_test)

    solution_metrics = calculate_metrics(
        exact,
        prediction
    )

    physics = physics_metrics(
        model,
        t_test
    )

    initial = initial_condition_metrics(
        model
    )

    result = {}

    result.update(solution_metrics)
    result.update(physics)
    result.update(initial)

    return result

In [16]:
t_test = torch.linspace(
    0.0,
    5.0,
    1000,
    dtype=torch.float32
).reshape(-1, 1)

metrics = evaluate_pinn(pinn, t_test)

for name, value in metrics.items():
    print(f"{name:25s}: {value:.10e}")

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.